## Observação:

### Essa videoula o instrutor passa muito rapidamente e ainda faz cortes no vídeo, além de código incompleto por vezes. Como eu não consegui rodar o código do instrutor, eu copiei esse código do card de outro estudante e apenas adicionei os comentários

In [ ]:
# baixa imagens
import os
if not os.path.exists('Faceswap-Deepfake-Pytorch'):
    !wget -q https://www.dropbox.com/s/5j17jlhts09ny/person_images.zip
    !wget -q https://raw.githubusercontent.com/sizhky/deep-fake-util/main/random_warp.py
!unzip -q person_images.zip
from torch_snippets import *
from random_warp import get_training_data

In [ ]:
import cv2
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# função para selecionar o rosto
def crop_face(img):

    # converte para cinza
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    # detecta o rosto, recorta ele e redimensiona
    if(len(faces)>0):
        for (x,y,w,h) in faces:
            img2 = img[y:(y+h),x:(x+w),:]
            img2 = cv2.resize(img2,(256,256))
            return img2, True
    else:
        return img, False

In [ ]:
# cria 2 pastas no diretório
!mkdir cropped_faces_personA
!mkdir cropped_faces_personB

# processa as fotos e salav na pasta
def crop_images(folder):
    images = Glob(folder+'/*.jpg')
    for i in range(len(images)):
        img = read(images[i],1)
        img2, face_detected = crop_face(img)
        if(face_detected==False):
            continue
        else:
            cv2.imwrite('cropped_faces_'+folder+'/'+str(i)+'.jpg',cv2.cvtColor(img2, cv2.COLOR_RGB2BGR))

crop_images('personA')
crop_images('personB')

In [ ]:
from torch.utils.data import Dataset,DataLoader

# criando o dataset personalizado e normalizado
class ImageDataset(Dataset):
    def __init__(self, items_A, items_B):

        # normalizaçao
        self.items_A = np.concatenate([read(f,1)[None] for f in items_A])/255.
        self.items_B = np.concatenate([read(f,1)[None] for f in items_B])/255.
        self.items_A += self.items_B.mean(axis=(0, 1, 2)) - self.items_A.mean(axis=(0, 1, 2))

    def __len__(self):
        return min(len(self.items_A), len(self.items_B))

    def __getitem__(self, ix):
        a, b = choose(self.items_A), choose(self.items_B)
        return a, b

    def collate_fn(self, batch):
        imsA, imsB = list(zip(*batch))
        # aplica distorções nas fotos
        imsA, targetA = get_training_data(imsA, len(imsA))
        imsB, targetB = get_training_data(imsB, len(imsB))
        imsA, imsB, targetA, targetB = [torch.Tensor(i).permute(0,3,1,2).to(device) for i in [imsA, imsB, targetA, targetB]]
        return imsA, imsB, targetA, targetB

# instannciando o dataset e o data loader
a = ImageDataset(Glob('cropped_faces_personA'), Glob('cropped_faces_personB'))
x = DataLoader(a, batch_size=32, collate_fn=a.collate_fn)

In [ ]:
inspect(*next(iter(x)))

for i in next(iter(x)):
    subplots(i[:8], nc=4, sz=(4,2))

In [ ]:
# bloco de convolução
def _convLayer(input_features, output_features):
    return nn.Sequential(
        nn.Conv2d(input_features, output_features, kernel_size=5, stride=2, padding=2),
        nn.LeakyReLU(0.1, inplace=True)
    )

# bloco de upscale (aumenta o tamanho da imagem, reconstrução do rosto)
def _UpScale(input_features, output_features):
    return nn.Sequential(
        nn.ConvTranspose2d(input_features, output_features, kernel_size=2, stride=2, padding=0),
        nn.LeakyReLU(0.1, inplace=True)
    )

#  redimensiona o tensor para passar de um bloco para o outro
class Reshape(nn.Module):
    def forward(self, input):
        output = input.view(-1, 1024, 4, 4) # channel * 4 * 4
        return output

In [ ]:
# classe para o autoencoder
class Autoencoder(nn.Module):
    def __init__(self,):
        super(Autoencoder, self).__init__()

        # encoder (comprime as caracteristicas mais importantes)
        self.encoder = nn.Sequential(
            _convLayer(3, 128),
            _convLayer(128, 256),
            _convLayer(256, 512),
            _convLayer(512, 1024),
            nn.Flatten(),
            nn.Linear(1024 * 4 * 4, 1024),
            nn.Linear(1024, 1024 * 4 * 4),
            Reshape(),
            _UpScale(1024, 512),
        )

        # decoder da pessoa A
        self.decoder_A = nn.Sequential(
            _UpScale(512, 256),
            _UpScale(256, 128),
            _UpScale(128, 64),
            nn.Conv2d(64, 3, kernel_size=3, padding=1),
            nn.Sigmoid(),
        )

        # decoder da pessoa B
        self.decoder_B = nn.Sequential(
            _UpScale(512, 256),
            _UpScale(256, 128),
            _UpScale(128, 64),
            nn.Conv2d(64, 3, kernel_size=3, padding=1),
            nn.Sigmoid(),
        )
    
        # troca de rostos
        def forward(self, x, select='A'):
            if select == 'A':
                out = self.encoder(x)
                out = self.decoder_A(out)
            else:
                out = self.encoder(x)
                out = self.decoder_B(out)
            return out